# FinGPT: Quantitative Trading Signal Terminal

This notebook demonstrates how to build, evaluate, and deploy a financial quantitative signal generation engine. It utilizes a fine-tuned DeepSeek model combined with a Milvus-powered Retrieval-Augmented Generation (RAG) system to analyze financial news and extract trading signals.

The workflow is divided into two main sections:
1. **Part 1: Local Setup, Evaluation, & Gradio UI** - Setting up the model, database, and a local web interface.
2. **Part 2: Cloud Deployment (GCP Cloud Run)** - Packaging the application with FastAPI and Docker to deploy it on Google Cloud.

## Part 1: Local Setup & Evaluation
In this section, we install the necessary libraries, load our fine-tuned DeepSeek model from Google Drive, and prepare the local Milvus vector database for our RAG architecture.

In [1]:
!pip install pymilvus

In [2]:
!pip install "datasets<3.0.0"

In [3]:
pip install -U openai fastapi uvicorn pydantic

In [4]:
!pip install milvus_lite

In [5]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
from google.colab import drive

# 1. Mount Google Drive (prevents file loss after disconnection)
drive.mount('/content/drive')

# 2. Path to DeepSeek weights
deepseek_lora_path = '/content/drive/MyDrive/finetuned_models/finetuned_deepseek-ai_DeepSeek-R1-Distill-Llama-8B'

print("📥 Reloading DeepSeek model into VRAM, please wait...")

# 3. Load base model
deepseek_base_model = AutoModelForCausalLM.from_pretrained(
    'deepseek-ai/DeepSeek-R1-Distill-Llama-8B',
    trust_remote_code=True,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

# 4. Merge trained LoRA weights
deepseek_model = PeftModel.from_pretrained(
    deepseek_base_model,
    deepseek_lora_path,
    torch_dtype=torch.bfloat16,
)
deepseek_model = deepseek_model.eval()

# 5. Load Tokenizer
deepseek_tokenizer = AutoTokenizer.from_pretrained('deepseek-ai/DeepSeek-R1-Distill-Llama-8B')
deepseek_tokenizer.padding_side = "right"
deepseek_tokenizer.pad_token = deepseek_tokenizer.eos_token

print("✅ DeepSeek model successfully reloaded!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📥 Reloading DeepSeek model into VRAM, please wait...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

✅ DeepSeek model successfully reloaded!


In [6]:
from datasets import load_dataset
import os

def load_quant_dataset(dataset_name, split="train", sample_size=None, save_path=None):
    """
    Unified dataset loading and formatting function

    Parameters:
    - dataset_name: Name of the dataset on Hugging Face
    - split: Split to load, e.g., "train", "validation", "test"
    - sample_size: Number of random samples to draw, loads all if None
    - save_path: Optional path to save to Google Drive
    """
    print(f"\n🌐 Preparing to load and process dataset: {dataset_name} ({split})...")

    # 1. Register dataset configuration (resolve field and label differences)
    configs = {
        "zeroshot/twitter-financial-news-sentiment": {
            "text_col": "text",
            "label_map": {0: "down", 1: "up", 2: "neutral"}
        },
        "FinanceInc/auditor_sentiment": {
            "text_col": "sentence",
            "label_map": {0: "down", 1: "neutral", 2: "up"}
        }
    }

    # Check if the dataset is supported
    if dataset_name not in configs:
        print(f"❌ Dataset not supported. Please configure mapping rules in configs first!")
        return None

    config = configs[dataset_name]

    try:
        # 2. Download and Sample
        raw_ds = load_dataset(dataset_name, split=split)

        if sample_size and sample_size < len(raw_ds):
            print(f"🎲 Randomly sampling {sample_size} records...")
            raw_ds = raw_ds.shuffle(seed=42).select(range(sample_size))

        # 3. Unified Formatting (Mapping)
        def format_func(ex):
            return {
                "prompt": ex[config["text_col"]],  # Read text column dynamically
                "answer": config["label_map"][ex["label"]] # Apply label rules dynamically
            }

        formatted_ds = raw_ds.map(format_func)

        # 4. Save to disk
        if save_path:
            formatted_ds.save_to_disk(save_path)
            print(f"💾 Data safely saved to: {save_path}")

        print(f"✅ Processing successful! Obtained {len(formatted_ds)} standardized records.")
        return formatted_ds

    except Exception as e:
        print(f"❌ Process interrupted! Error message: {e}")
        return None

In [7]:
from datasets import concatenate_datasets

print("Initiating ultimate Dev/Test pipeline...")

# 1. Load Data Sources (Sample 100 from Twitter, full ~100 from Auditor)
auditor_full = load_quant_dataset("FinanceInc/auditor_sentiment", split="train", sample_size=100)
twitter_subset = load_quant_dataset("zeroshot/twitter-financial-news-sentiment", split="validation", sample_size=100)

# 2. Split in half separately (ensure Twitter and Auditor ratio is always 1:1 in both sets)
twitter_splits = twitter_subset.train_test_split(test_size=0.5, seed=42)
auditor_splits = auditor_full.train_test_split(test_size=0.5, seed=42)

# 3. Assemble raw Dev and Test sets, then shuffle
dev_ds_raw = concatenate_datasets([twitter_splits['train'], auditor_splits['train']]).shuffle(seed=42)
test_ds_raw = concatenate_datasets([twitter_splits['test'], auditor_splits['test']]).shuffle(seed=42)


# 4. Unify and drop historical baggage (redundant columns)!
cols_to_remove = ["text", "sentence", "label"]
dev_ds = dev_ds_raw.remove_columns(cols_to_remove)
test_ds = test_ds_raw.remove_columns(cols_to_remove)

# 5. Verify Results
print(f"🛠️ Error Analysis Set (Dev Set): {len(dev_ds)} records")
print("👀 Dev Set sample:", dev_ds[0])

print(f"\n🎓 Blind Test Set (Test Set): {len(test_ds)} records")
print("👀 Test Set sample:", test_ds[0])

Initiating ultimate Dev/Test pipeline...

🌐 Preparing to load and process dataset: FinanceInc/auditor_sentiment (train)...
🎲 Randomly sampling 100 records...
✅ Processing successful! Obtained 100 standardized records.

🌐 Preparing to load and process dataset: zeroshot/twitter-financial-news-sentiment (validation)...
🎲 Randomly sampling 100 records...
✅ Processing successful! Obtained 100 standardized records.
🛠️ Error Analysis Set (Dev Set): 100 records
👀 Dev Set sample: {'prompt': 'Following the acquisitions , Panostaja will establish a new business unit which will focus on heat treatment of metals .', 'answer': 'neutral'}

🎓 Blind Test Set (Test Set): 100 records
👀 Test Set sample: {'prompt': "`` The industry is coming to an interesting fork in the road as both handset manufacturers and wireless carriers attempt to serve as the portal for Web-based service to your wireless handset , '' he wrote .", 'answer': 'neutral'}


In [8]:
from tqdm.auto import tqdm
def evaluate_quant_model(generator, dataset, sample_size=20):
    """
    Quantitative Large Model Signal Backtesting Evaluation Function

    Parameters:
    - generator: Instantiated signal generator (e.g., our rag_generator)
    - dataset: Hugging Face format test dataset
    - sample_size: Number of data points for backtesting

    Returns:
    - preds: List of predicted directions from the model [1, 0, -1, ...]
    - gts: List of actual stock directions [1, 0, -1, ...]
    """
    preds = []
    gts = []

    # Ensure the set sample size does not exceed the total dataset length
    actual_size = min(sample_size, len(dataset))

    # Subset for testing
    test_subset = dataset.select(range(actual_size))

    print(f"📊 Starting quantitative signal backtest, testing {actual_size} news items...")

    # Wrap the loop with tqdm for a nice progress bar
    for i in tqdm(range(actual_size), desc="Backtesting"):
        item = test_subset[i]
        prompt = item["prompt"]
        raw_answer = item["answer"]

        # 1. Ground Truth Parsing and Standardization
        # Automatically compatible with text labels ("up", "down") and numeric labels (1, -1)
        gt_direction = 0
        ans_str = str(raw_answer).strip().lower()

        if ans_str in ["up", "positive", "1", "1.0"]:
            gt_direction = 1
        elif ans_str in ["down", "negative", "-1", "-1.0"]:
            gt_direction = -1
        elif ans_str in ["neutral", "0", "0.0"]:
            gt_direction = 0
        else:
            # If it's another unknown format, try to convert directly to an integer
            try:
                gt_direction = int(float(raw_answer))
            except ValueError:
                gt_direction = 0

        gts.append(gt_direction)

        # 2. Model Inference (Prediction)
        try:
            # get JSON signal
            signal = generator.generate_signal(prompt)
            # Extract the 'direction' field from JSON, default to 0 if extraction fails
            pred_direction = signal.get("direction", 0)
        except Exception as e:
            # Prevent the entire backtest from stopping due to an error in one data point
            print(f"⚠️ Inference error for item {i}: {e}")
            pred_direction = 0

        preds.append(pred_direction)

    # 3. Calculate Win Rate and Print Results
    correct = sum(1 for p, g in zip(preds, gts) if p == g)
    accuracy = correct / actual_size if actual_size > 0 else 0

    print("\n" + "=" * 45)
    print(f"🏆 DeepSeek Quant Signal Prediction Accuracy: {accuracy * 100:.2f}%")
    print("=" * 45)

    # Return preds and gts
    return preds, gts

In [9]:
import os
import shutil
from google.colab import drive
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
from pymilvus import MilvusClient

# Step 0: Mount Drive and Prepare Paths
print("☁️ [0/5] Connecting to Google Drive...")
drive.mount('/content/drive')

db_folder = "/content/drive/MyDrive/quant_data"
drive_db_path = f"{db_folder}/financial_rag.db"
local_db_path = "/content/financial_rag.db"

# Ensure storage directory exists
os.makedirs(db_folder, exist_ok=True)
print(f"📁 Ensuring storage directory exists: {db_folder}")

# Step 1: Load Model
print("📥 [1/5] Loading Embedding Model...")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Step 2: Download and Clean Financial Rules Data
print("📚 [2/5] Downloading and cleaning financial rules data...")
dataset = load_dataset(
    "financial_phrasebank",
    "sentences_allagree",
    split='train',
    trust_remote_code=True
)
label_map = {0: "Down", 1: "Neutral", 2: "Up"}

# Extract and deduplicate
rules = list(set([
    f"Financial Logic: {entry['sentence']} | Direction: {label_map[entry['label']]}"
    for entry in dataset
]))
print(f"✅ Successfully extracted {len(rules)} deduplicated professional rules!")

# Step 3: Convert Text to High-Dimensional Vectors
print("🧮 [3/5] Converting text to high-dimensional vectors (takes about ten seconds)...")
rule_embeddings = embedding_model.encode(rules)

# Step 4: Store in Milvus Database Locally
print(f"🚀 [4/5] Starting Milvus engine in local fast environment: {local_db_path}")
client = MilvusClient(local_db_path) # Connect to local DB!

collection_name = "finance_rules"
dimension = rule_embeddings.shape[1]

# Clear old collection, create new one
if client.has_collection(collection_name):
    client.drop_collection(collection_name)

client.create_collection(
    collection_name=collection_name,
    dimension=dimension
)

# Assemble data and batch insert
data = []
for i, (rule, emb) in enumerate(zip(rules, rule_embeddings)):
    data.append({
        "id": i,
        "vector": emb.tolist(),
        "text": rule
    })

res = client.insert(collection_name=collection_name, data=data)
print(f"📊 Successfully stored {res['insert_count']} financial logics locally.")

# Step 5: Backup to Drive
print("💾 [5/5] Backing up local database to Google Drive...")
shutil.copy2(local_db_path, drive_db_path) # ⚠️ Key change: Physical copy!

print("="*50)
print(f"🎉 Mission accomplished! PyMilvus knowledge base built and backed up!")
print(f"💾 Database safely saved to your Google Drive: {drive_db_path}")
print("="*50)

☁️ [0/5] Connecting to Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📁 Ensuring storage directory exists: /content/drive/MyDrive/quant_data
📥 [1/5] Loading Embedding Model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


📚 [2/5] Downloading and cleaning financial rules data...
✅ Successfully extracted 2259 deduplicated professional rules!
🧮 [3/5] Converting text to high-dimensional vectors (takes about ten seconds)...
🚀 [4/5] Starting Milvus engine in local fast environment: /content/financial_rag.db
📊 Successfully stored 2259 financial logics locally.
💾 [5/5] Backing up local database to Google Drive...
🎉 Mission accomplished! PyMilvus knowledge base built and backed up!
💾 Database safely saved to your Google Drive: /content/drive/MyDrive/quant_data/financial_rag.db


In [10]:
# Count the number of deduplicated samples for the three cases
up_count = sum(1 for rule in rules if "Direction: Up" in rule)
down_count = sum(1 for rule in rules if "Direction: Down" in rule)
neutral_count = sum(1 for rule in rules if "Direction: Neutral" in rule)

print(f"✅ Successfully extracted {len(rules)} deduplicated professional rules!")
print("-" * 40)
print(f"📈 Bullish (Up) Samples   : {up_count} records")
print(f"📉 Bearish (Down) Samples : {down_count} records")
print(f"⚖️ Neutral Samples      : {neutral_count} records")
print("-" * 40)

✅ Successfully extracted 2259 deduplicated professional rules!
----------------------------------------
📈 Bullish (Up) Samples   : 570 records
📉 Bearish (Down) Samples : 303 records
⚖️ Neutral Samples      : 1386 records
----------------------------------------


In [11]:
import json
import re
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from pymilvus import MilvusClient

# RAG Signal Generator
class RAGFinGPTSignalGenerator:
    def __init__(self, model, tokenizer, embedding_model, db_client=None):
        self.model = model
        self.tokenizer = tokenizer
        self.embedding_model = embedding_model
        self.db_client = db_client

    def _dehydrate_news(self, long_text):
        """Long text information dehydrator"""
        print("🧽 Dehydrating long text information...")
        extraction_prompt = f"""You are a senior financial analyst. Read the following article and extract ONLY the hard financial, operational, and strategic facts.
        Ignore all CEO quotes, ESG statements, and PR fluff.
        Output the facts as a single concise paragraph.

        Article:
        {long_text}

        Extracted Financial Facts:"""

        inputs = self.tokenizer(extraction_prompt, return_tensors="pt").to(self.model.device)
        outputs = self.model.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.1,
            do_sample=False
        )
        raw_output = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        facts_text = raw_output.split("Extracted Financial Facts:")[-1].strip()
        print(f"💧 Dehydration complete! Core facts: {facts_text}")

        return facts_text

    def _search_knowledge_base(self, query_text):
        """Milvus vector retrieval"""
        try:
            query_vec = self.embedding_model.encode([query_text])[0].tolist()
            search_res = self.db_client.search(
                collection_name="finance_rules",
                data=[query_vec],
                limit=2,
                output_fields=["text"]
            )
            context_list = []
            if search_res and len(search_res[0]) > 0:
                for hit in search_res[0]:
                    if hit['distance'] > 0.4:
                        context_list.append(hit['entity']['text'])
            if context_list:
                return "\n".join(context_list)
            else:
                return "No specific financial standard matched. Proceeding with LLM intuition."
        except Exception as e:
            print(f"❌ RAG retrieval exception: {e}")
            return "Knowledge base offline. Relying purely on internal parameters."

    def _parse_output(self, text):
        """Powerful Markdown parsing + JSON parsing"""
        cleaned_text = text.replace("```json", "").replace("```", "").strip()
        try:
            matches = re.findall(r'\{.*?\}', cleaned_text, re.DOTALL)
            if matches:
                final_json_str = matches[-1].replace("'", '"')
                signal_dict = json.loads(final_json_str)
                return {
                    "direction": int(signal_dict.get("direction", 0)),
                    "confidence": int(signal_dict.get("confidence", 0))
                }
        except Exception as e:
            print(f"❌ Error parsing JSON: {e}")
        return {"direction": 0, "confidence": 0}

    def generate_signal(self, news_text):
        """
        Signal generation entry: integrates dehydration, RAG, Prompt construction, and Chain of Thought extraction
        """
        # 1. Smart Dehydration (if the text is too long)
        if len(news_text.split()) > 80 or len(news_text) > 500:
            core_news = self._dehydrate_news(news_text)
        else:
            core_news = news_text

        # 2. Vector database retrieval
        retrieved_context = self._search_knowledge_base(core_news)

        # 3. Assemble Ultimate Prompt
        system_prompt = f"""You are a strict Financial Sentiment Classifier.
Analyze the text and apply ONE of the following rules.

RULE 1: NEGATIVE TRIGGERS (Return -1)
IF the text contains:
- Worsening financials: "down", "decreased", "loss widened", "declining"
- Comparisons showing a drop: "lower compared to", "decreased from"
- Macro/Operational risks: "crisis", "strike", "layoffs", "stopped", "closed"
THEN PREDICT -1.

RULE 2: POSITIVE TRIGGERS (Return 1)
IF the text contains:
- Improving financials: "growth", "improved by", "loss narrowed", "increased", "grew", "Positive cash flow"
- Comparisons showing a rise: "higher compared to", "increased from"
- Contracts & Sentiment: "won the contract", "strategic cooperation", "confidence", "satisfied", "successful"
THEN PREDICT 1.

RULE 0: THE NEUTRAL FILTER (Return 0)
ONLY apply this if Rules 1 and 2 DO NOT MATCH.
IF the text is purely:
- PR fluff: "proud of our team", "ESG goals", "global leader"
- Standalone numbers WITHOUT any historical comparison: "Revenue is $10M" (and no mention of previous years).
THEN PREDICT 0.

RAG Context: {retrieved_context}

Instructions:
1. You MUST start with <think>.
2. Step 1: Read the RAG Context. If it explicitly states a Direction (e.g., Direction: Up/1), state this.
3. Step 2: Read the News. Look for explicit comparison words ("compared to", "improved by", "narrowed").
4. Step 3: Match with Rule 1, 2, or 0.
5. Close with </think> and output valid JSON ONLY: {{"direction": 1, "confidence": 95}}.

[Example]
News: "Revenue was $412M. Net loss compared to last year improved by $32.5M. Cash flow is Positive."
Result:
<think>
Step 1: RAG context not provided in this example.
Step 2: The news contains "improved by $32.5M" and "Positive cash flow". These are explicit comparison and growth markers.
Step 3: This perfectly matches the positive triggers in Rule 2 ("improved by", "Positive cash flow"). Rule 0 is skipped because comparisons exist.
Conclusion: Rule 2 applies. Prediction is 1.
</think>
{{"direction": 1, "confidence": 90}}

[Now analyze the following]
News: {core_news}
Result:
<think>
Step 1: Evaluate RAG Context.
"""

        # 4. LLM Inference
        inputs = self.tokenizer(system_prompt, return_tensors="pt").to(self.model.device)
        outputs = self.model.generate(
            **inputs,
            max_new_tokens=800,
            temperature=0.1,
            do_sample=False,
            pad_token_id=self.tokenizer.eos_token_id
        )

        input_length = inputs.input_ids.shape[1]
        generated_tokens = outputs[0][input_length:]
        raw_output = self.tokenizer.decode(generated_tokens, skip_special_tokens=False)
        raw_output = raw_output.replace("<｜end of sentence｜>", "").replace("<|eot_id|>", "").replace("<eos>", "").strip()

        print(f"--- DEBUG RAW OUTPUT ---\n{raw_output}\n------------------------")

        # 5. Extract Chain of Thought
        think_content = ""
        if "<think>" in raw_output and "</think>" in raw_output:
            think_content = raw_output.split("<think>")[1].split("</think>")[0].strip()
        elif "</think>" in raw_output:
            think_content = raw_output.split("</think>")[0].replace("<think>", "").strip()
        else:
            think_content = raw_output.split("{")[0].replace("<think>", "").strip()

        if not think_content:
            think_content = "The model directly provided a conclusion without outputting intermediate reasoning."

        # 6. Use parsing function to extract final signal
        final_signal = self._parse_output(raw_output)

        # 7. Package and return to Gradio frontend
        return {
            "signal": final_signal,
            "think": think_content,
            "context": retrieved_context,
            "dehydrated_news": core_news,
            "raw_output": raw_output
        }

# Instantiation and Launch
client = MilvusClient("/content/drive/MyDrive/quant_data/financial_rag.db")
rag_generator = RAGFinGPTSignalGenerator(
    model=deepseek_model,
    tokenizer=deepseek_tokenizer,
    embedding_model=embedding_model,
    db_client=client
)
print("✅ Engine assembly complete, Gradio can be launched!")

✅ Engine assembly complete, Gradio can be launched!


In [12]:
# print("\n🚀 Running backtest with lightweight RAG architecture...")
# Run the backtest
# preds, gts = evaluate_quant_model(rag_generator, dev_ds, sample_size=10)

In [ ]:
# print("🔍 Starting Error Analysis...\n")

# error_count = 0
# for i in range(len(preds)):
    # if preds[i] != gts[i]:
        # error_count += 1
        # # Get the original text and corresponding label
        # original_text = dev_ds[i]['prompt']

        # print(f"❌ Error #{error_count}")
        # print(f"📰 Original News: {original_text}")
        # print(f"🤖 Model Prediction: {preds[i]}  |  ✅ Ground Truth: {gts[i]}")
        # print("-" * 50)

# print(f"\n💡 Out of 100 questions, {error_count} were wrong.")

In [ ]:
# print("\n🚀 Final accuracy...")
# Execute backtest
# preds, gts = evaluate_quant_model(rag_generator, test_ds, sample_size=100)

In [13]:
import gradio as gr
import re
import json

# 1. Bridge Function: Pure text processing + Extract 4 major outputs
def analyze_news(news_text):
    # 1. Input validation
    if not news_text or not news_text.strip():
        return "⚠️ Waiting for input", "0%", "N/A", "Please enter the financial news article to be analyzed on the left."
    try:
        # 2. Call the ultimate generator for inference
        # news_text will automatically go through: long text dehydration -> RAG retrieval -> LLM inference
        result = rag_generator.generate_signal(news_text)

        # 3. Parse the generated dictionary
        signal = result.get("signal", {})
        direction = signal.get("direction", 0)
        confidence = signal.get("confidence", 0)
        think_content = result.get("think", "Model did not output thought process")
        retrieved_context = result.get("context", "No relevant background found")

        # 4. Format for UI display
        if direction == 1:
            dir_str = "📈 Bullish (1)"
        elif direction == -1:
            dir_str = "📉 Bearish (-1)"
        else:
            dir_str = "⚖️ Neutral (0)"

        dehydrated_text = result.get("dehydrated_news", news_text)

        # Return 5 variables!
        return dir_str, f"{confidence}%", retrieved_context, think_content, dehydrated_text

    except Exception as e:
        print(f"UI runtime error: {e}")
        return "❌ Runtime Error", "0%", "N/A", f"Underlying runtime error: {str(e)}"

# 2. Gradio UI
with gr.Blocks(theme=gr.themes.Soft(), title="DeepSeek Quant Terminal") as demo:
    gr.Markdown(
        """
        <div style='text-align: center;'>
        <h1>🚀 FinGPT & DeepSeek Quantitative Signal Terminal</h1>
        <p>Current underlying engine: <b>DeepSeek-R1-Distill-Llama-8B (LoRA Fine-tuned) + Knowledge Base Enhanced</b></p>
        </div>
        """ # Change Markdown text
    )

    with gr.Row():
        # Left: Input Area
        with gr.Column(scale=1, variant="panel"):
            gr.Markdown("### 🎛️ Signal Control Panel") # Change here

            # Component 1: Textbox
            news_input = gr.Textbox(
                label="📰 Enter Financial News Article", # Change here
                placeholder="Paste financial news or long financial report paragraphs here (long text will be automatically dehydrated)...", # Change here
                lines=10
            )

            analyze_btn = gr.Button("🚀 Execute DeepSeek Deep Inference", variant="primary") # Change here

            gr.Markdown("---")
            gr.Examples(
                examples=[
                    ["Following the acquisitions , Panostaja will establish a new business unit which will focus on heat treatment of metals"],
                    ["Wartsila has signed a 5-year strategic agreement to supply engine components to a major Singaporean shipping line."],
                    ["The company reported a net loss of $12.5 million this quarter, but the board of directors expressed satisfaction with the current progress of the restructuring program."]
                ],
                inputs=[news_input]
            )

        # Right: Result Display Area
        with gr.Column(scale=2):
            gr.Markdown("### 📊 Real-time Quantitative Analysis") # Change here
            with gr.Row():
                # Output 1 & 2
                direction_out = gr.Label(label="Predicted Direction") # Change here
                confidence_out = gr.Textbox(label="🎯 Signal Confidence", scale=1) # Change here

            # Output 3: RAG
            with gr.Accordion("📚 Knowledge Base Reference (RAG Context)", open=False): # Change here
                rag_out = gr.Textbox(label="Hit Financial Audit Rules", lines=3) # Change here

            # Output 4: Chain of Thought
            with gr.Accordion("🧠 DeepSeek Internal Logic Chain of Thought (XAI)", open=True): # Change here
                cot_out = gr.Textbox(
                    label="Reasoning Process (Chain of Thought)", # Change here
                    lines=14,
                    placeholder="DeepSeek's thought process will be displayed here..." # Change here
                )
            with gr.Accordion("💧 LLM Information Dehydration Result (Triggered only for long text)", open=False): # Change here
                dehydrated_out = gr.Textbox(label="Cleaned facts input to the core engine", lines=4) # Change here

    # Bind Events
    # ⚠️ Restore single input stream, clear and concise
    analyze_btn.click(
        fn=analyze_news,
        inputs=[news_input],
        outputs=[direction_out, confidence_out, rag_out, cot_out, dehydrated_out] # Now 5 outputs
    )
# Launch
if __name__ == "__main__":
    demo.launch(share=True, debug=True)

/tmp/ipykernel_9554/1875205425.py:40: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="DeepSeek Quant Terminal") as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://9fc2ddf6888e7e1437.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://9fc2ddf6888e7e1437.gradio.live


## Part 2: Cloud Deployment (GCP Cloud Run)
Now that our local engine and Gradio UI are working correctly, we will transition to cloud deployment.

In this section, we will:
1. Install deployment dependencies (`FastAPI`, `Uvicorn`, etc.).
2. Prepare the Dockerfile and Python application scripts (`main.py`, `config_prompts.py`).
3. Authenticate with Google Cloud and deploy the containerized application to Google Cloud Run.

In [14]:
!cp /content/drive/MyDrive/quant_data/financial_rag.db ./financial_rag.db
print("✅ DB file ready, waiting for packaging")

✅ DB file ready, waiting for packaging


In [15]:
from google.colab import auth
auth.authenticate_user()
print("✅ Login successful!")

✅ Login successful!


In [16]:
%%writefile requirements.txt
fastapi
uvicorn
pydantic
openai
sentence-transformers
pymilvus
milvus-lite
gradio

Writing requirements.txt


In [17]:
%%writefile Dockerfile

# 1. Discard 'slim' and use the full base image, which includes all necessary underlying C++ libraries
FROM python:3.10

# 2. Set the working directory
WORKDIR /app

# 3. Install dependencies first (leverage Docker cache layers to speed up builds)
COPY requirements.txt .
RUN pip install --no-cache-dir torch --index-url https://download.pytorch.org/whl/cpu
RUN pip install --no-cache-dir -r requirements.txt

# 4. Pre-download the Embedding model during the build phase!
# This ensures the container starts instantly and avoids startup timeout errors
RUN python -c "from sentence_transformers import SentenceTransformer; SentenceTransformer('all-MiniLM-L6-v2')"

# 5. Copy code and vector database files
COPY . .

# 6. Use the safest and most standard Uvicorn startup command, binding strictly to port 8080
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8080"]

Writing Dockerfile


In [18]:
%%writefile config_prompts.py

def get_financial_prompt(retrieved_context):
    """
    Generates a highly structured financial analysis prompt.
    Injects RAG retrieved context.
    """
    return f"""You are a strict Financial Sentiment Classifier.
Analyze the text and apply ONE of the following rules.

RULE 1: NEGATIVE TRIGGERS (Return -1)
- Worsening financials: "down", "decreased", "loss widened", "declining"
- Comparisons showing a drop: "lower compared to", "decreased from"
- Macro/Operational risks: "crisis", "strike", "layoffs", "closed"
THEN PREDICT -1.

RULE 2: POSITIVE TRIGGERS (Return 1)
- Improving financials: "growth", "improved by", "loss narrowed", "increased"
- Contracts & Sentiment: "won the contract", "strategic cooperation", "successful"
THEN PREDICT 1.

RULE 0: THE NEUTRAL FILTER (Return 0)
ONLY apply if Rule 1 & 2 DO NOT MATCH. (e.g., PR fluff or standalone numbers without comparison).

RAG Context: {retrieved_context}

Instructions:
1. You MUST start with <think>.
2. Step 1: Evaluate RAG Context and its Direction.
3. Step 3: Final match with Rule 1, 2, or 0.
4. End with </think> and output valid JSON ONLY: {{\"direction\": 1, \"confidence\": 95, \"note\": \"reason\"}}.
"""

Writing config_prompts.py


In [19]:
%%writefile main.py
import os, json, re
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from openai import OpenAI
from sentence_transformers import SentenceTransformer
from pymilvus import MilvusClient
import gradio as gr

# Import prompt configuration module
from config_prompts import get_financial_prompt

# 1. Global Initialization
DEEPSEEK_API_KEY = os.environ.get("DEEPSEEK_API_KEY")
client = OpenAI(api_key=DEEPSEEK_API_KEY, base_url="https://api.deepseek.com")

try:
    model = SentenceTransformer('all-MiniLM-L6-v2')
    db = MilvusClient("./financial_rag.db")
except Exception as e:
    print(f"Init Error: {e}")
    model, db = None, None

app = FastAPI()

class NewsInput(BaseModel):
    news_text: str

# 2. Core Analysis Function (called by both API and UI)
def run_core_analysis(news_text: str):
    core_news = news_text[:1000]

    # RAG retrieval
    retrieved_context = "No specific rules found in the current knowledge base."
    if db and model:
        try:
            query_vec = model.encode([core_news])[0].tolist()
            search_res = db.search(collection_name="finance_rules", data=[query_vec], limit=1, output_fields=["text"])
            if search_res and len(search_res[0]) > 0:
                retrieved_context = search_res[0][0]['entity'].get('text', retrieved_context)
        except Exception:
            pass

    system_prompt = get_financial_prompt(retrieved_context)

    # Call the LLM
    response = client.chat.completions.create(
        model="deepseek-reasoner",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"News: {core_news}\nResult:"}
        ]
    )

    think = getattr(response.choices[0].message, 'reasoning_content', "")
    full_content = response.choices[0].message.content

    # Parse JSON
    json_match = re.search(r'\{.*\}', full_content, re.DOTALL)
    signal = json.loads(json_match.group().replace("'", '"')) if json_match else {"direction": 0, "confidence": 0, "note": "Parse error"}

    return {
        "final_signal": signal,
        "think_process": think if think else "No reasoning process captured.",
        "context": retrieved_context,
        "dehydrated_news": core_news
    }

# 3. FastAPI Route (for programmatic access)
@app.post("/api/v1/analyze")
def analyze_api(input_data: NewsInput):
    try:
        data = run_core_analysis(input_data.news_text)
        return {"status": "success", "data": data}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# 4. Gradio Web UI (for human access)
def analyze_ui(news_text):
    if not news_text or not news_text.strip():
        return "⚖️ Neutral (0)", "0%", "N/A", "Please enter financial news.", "N/A"
    try:
        # The web page directly calls the core function, no network request needed, faster!
        data = run_core_analysis(news_text)

        sig = data.get("final_signal", {})
        direction = sig.get("direction", 0)

        dir_str = "📈 Bullish (1)" if direction == 1 else "📉 Bearish (-1)" if direction == -1 else "⚖️ Neutral (0)"
        conf_str = f"{sig.get('confidence', 0)}%"

        return dir_str, conf_str, data.get("context"), data.get("think_process"), data.get("dehydrated_news")
    except Exception as e:
        return "❌ Error", "0%", "N/A", f"Processing Error: {str(e)}", "N/A"

with gr.Blocks(theme=gr.themes.Soft(), title="DeepSeek Quant Terminal") as demo:
    gr.Markdown(
        """
        <div style='text-align: center;'>
        <h1>🚀 FinGPT & DeepSeek Quantitative Signal Terminal</h1>
        <p>Backend Status: <b>Cloud Run (us-east4) Active</b> | Engine: <b>DeepSeek-R1-Reasoner</b></p>
        </div>
        """
    )
    with gr.Row():
        with gr.Column(scale=1, variant="panel"):
            gr.Markdown("### 🎛️ Signal Control Panel")
            news_input = gr.Textbox(label="📰 Enter Financial News Article", placeholder="Paste news here...", lines=10)
            analyze_btn = gr.Button("🚀 Execute DeepSeek Deep Inference", variant="primary")
            gr.Markdown("---")
            gr.Examples(
                examples=[
                    ["NVIDIA reports record revenue, beating expectations with strong AI chip demand."],
                    ["The company announced a potential bankruptcy filing following a failed debt restructuring."],
                    ["Market remains steady as the Federal Reserve keeps interest rates unchanged."]
                ],
                inputs=[news_input]
            )
        with gr.Column(scale=2):
            gr.Markdown("### 📊 Real-time Quantitative Analysis")
            with gr.Row():
                direction_out = gr.Label(label="Predicted Direction")
                confidence_out = gr.Textbox(label="🎯 Signal Confidence", scale=1)
            with gr.Accordion("📚 Knowledge Base Reference (RAG Context)", open=False):
                rag_out = gr.Textbox(label="Hit Financial Audit Rules", lines=3)
            with gr.Accordion("🧠 DeepSeek Internal Logic Chain of Thought (XAI)", open=True):
                cot_out = gr.Textbox(label="Reasoning Process (Chain of Thought)", lines=14)
            with gr.Accordion("💧 LLM Information Dehydration Result", open=False):
                dehydrated_out = gr.Textbox(label="Cleaned facts", lines=4)

    analyze_btn.click(
        fn=analyze_ui,
        inputs=[news_input],
        outputs=[direction_out, confidence_out, rag_out, cot_out, dehydrated_out]
    )

# 🌟 Mount Gradio at the root directory "/" of FastAPI
app = gr.mount_gradio_app(app, demo, path="/")

Writing main.py


In [20]:
# 1. Create a clean folder
!mkdir -p /content/fingpt_project

# 2. Move the 4 core files into this folder
!mv /content/main.py /content/fingpt_project/
!mv /content/requirements.txt /content/fingpt_project/
!mv /content/Dockerfile /content/fingpt_project/
!mv /content/financial_rag.db /content/fingpt_project/
!mv /content/config_prompts.py /content/fingpt_project/

# 3. Change Colab's working directory to this folder
%cd /content/fingpt_project

# 4. Verify that only these 4 files are in the folder
!ls -l

/content/fingpt_project
total 4584
-rw-r--r-- 1 root root    1187 Apr 27 11:22 config_prompts.py
-rw-r--r-- 1 root root     876 Apr 27 11:22 Dockerfile
-rw-r--r-- 1 root root 4673536 Apr 27 11:22 financial_rag.db
-rw-r--r-- 1 root root    5639 Apr 27 11:22 main.py
-rw-r--r-- 1 root root      82 Apr 27 11:22 requirements.txt


In [21]:
!gcloud projects add-iam-policy-binding project-72f106cd-df78-4301-a75 --member="serviceAccount:250441276906-compute@developer.gserviceaccount.com" --role="roles/storage.admin" && \
gcloud projects add-iam-policy-binding project-72f106cd-df78-4301-a75 --member="serviceAccount:250441276906-compute@developer.gserviceaccount.com" --role="roles/logging.logWriter" && \
gcloud projects add-iam-policy-binding project-72f106cd-df78-4301-a75 --member="serviceAccount:250441276906-compute@developer.gserviceaccount.com" --role="roles/artifactregistry.writer"

Updated IAM policy for project [project-72f106cd-df78-4301-a75].
bindings:
- members:
  - serviceAccount:service-250441276906@gcp-sa-artifactregistry.iam.gserviceaccount.com
  role: roles/artifactregistry.serviceAgent
- members:
  - serviceAccount:250441276906-compute@developer.gserviceaccount.com
  role: roles/artifactregistry.writer
- members:
  - serviceAccount:250441276906@cloudbuild.gserviceaccount.com
  role: roles/cloudbuild.builds.builder
- members:
  - serviceAccount:service-250441276906@gcp-sa-cloudbuild.iam.gserviceaccount.com
  role: roles/cloudbuild.serviceAgent
- members:
  - serviceAccount:service-250441276906@containerregistry.iam.gserviceaccount.com
  role: roles/containerregistry.ServiceAgent
- members:
  - serviceAccount:250441276906-compute@developer.gserviceaccount.com
  role: roles/logging.logWriter
- members:
  - user:gavinzhang588@gmail.com
  role: roles/owner
- members:
  - serviceAccount:service-250441276906@gcp-sa-pubsub.iam.gserviceaccount.com
  role: roles/

In [22]:
import os
from google.colab import userdata

# Google Cloud Run One-Click Deployment Script
PROJECT_ID = userdata.get('GCP_PROJECT_ID')
DEEPSEEK_KEY = userdata.get('DEEPSEEK_API_KEY')

# Set Google Cloud Project ID
!gcloud config set project {PROJECT_ID}

# Enable Cloud Build and Run services
!gcloud services enable run.googleapis.com cloudbuild.googleapis.com

# Start packaging and deploying
!gcloud run deploy fingpt-api \
  --source . \
  --region us-east4 \
  --allow-unauthenticated \
  --quiet \
  --memory=4Gi \
  --cpu=2 \
  --timeout=600 \
  --execution-environment=gen2 \
  --set-env-vars="DEEPSEEK_API_KEY={DEEPSEEK_KEY}"

[environment: untagged] Read more to tag: g.co/cloud/project-env-tag.
Updated property [core/project].
Operation "operations/acat.p2-250441276906-96a6a570-361d-4bcf-96e6-ef98e96d8b1f" finished successfully.
Building using Dockerfile and deploying container to Cloud Run service [fingpt-api] in project [project-72f106cd-df78-4301-a75] region [us-east4]
Service [fingpt-api] revision [fingpt-api-00018-w6d] has been deployed and is serving 100 percent of traffic.
Service URL: https://fingpt-api-250441276906.us-east4.run.app
